# Lead Qualifier

Tool which classifies whether a company is a good sales lead when given a business website.

In [1]:
import os, requests, ollama, json
from dotenv import load_dotenv
from openai import OpenAI
from scraper import fetch_website_links, fetch_website_contents
from IPython.display import Markdown, display, update_display

In [2]:
load_dotenv(override=True)

LLAMA_MODEL = 'llama3.2:1b'
OLLAMA_BASE_URL = "http://localhost:11434/v1"

llama = OpenAI(base_url=OLLAMA_BASE_URL, api_key=ollama)

In [3]:
requests.get("http://localhost:11434").content

b'Ollama is running'

steps: 
- pull landing web page data 
- pull links from landing web page.  
- pull landing web page data and data from relevant web pages
- analyse whether company is a good sales lead for your company


In [20]:
system_prompt = """
You are a helpful assistant that will summarise the relevant offerings from a company from its website. 
Respond in markdown.
Ignore any navigation data, terms of service, privacy policy, or any other non-company related data.
"""

user_prompt = """
Here are the contents of a website.
Provide a short summary of this website.
"""

In [ ]:
# system_prompt = """
# You are a helpful assistant that will gather the relevant information about a company from its website. 

# You will be given a website url, and you will need to extract the following information:
# - Company Name
# - Company Description
# - Company Industry
# - Company offerings (products/services)
# - Company Expenses (products/services)
# - Company Revenue
# - Company Pain Points
# - Company Budget
# - Company Employees
# - Company Location

# Ignore any navigation data, terms of service, privacy policy, or any other non-company related data.
# """

# user_prompt = """
# Please give me a comprehensive breakdown of this company and their products/services. 
# Also, give me a list of what services and products they would be interested in.
# """

In [15]:
def get_website_data(url):
    website = fetch_website_contents(url)
    response = llama.chat.completions.create(
        model=LLAMA_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt + website}
        ]
    )
    return response.choices[0].message.content

In [23]:
get_website_data('https://www.bishopsgatedental.co.uk/about-us/')

'### Bishopsgate Dental Care: A Comprehensive Dentistry Offering a Range of Services\n\nBishopsgate Dental Care is a dental practice that provides various services, including cosmetic dentistry, general dentistry, and orthodontics. They offer specialized treatments such as Invisalign, Invisalign Clear Aligners for teens, veneers, composite bonding, teeth whitening, and more.\n\n#### Key Features:\n\n*   Meet Our Team: Learn about the team members involved in delivering exceptional patient care.\n*   Awards & Recognition: Check the accomplishments of Bishopsgate Dental Care winners.\n*   Testimonials & Reviews: Read from genuine patients who have utilized services at this dental practice.\n*   Vivera Retainers, Cosmetic Dentistry for Teeth Whitening and more'

In [27]:
fetch_website_links('https://www.bishopsgatedental.co.uk/about-us/')

['#content',
 'tel:02073776762',
 '/contact-us/',
 'tel:02073776762',
 'https://uk.dentalhub.online/soe/new/Bishopsgate%20Dental%20Care?pid=UKMAD03',
 '/contact-us/',
 'https://www.bishopsgatedental.co.uk/about-us/',
 'https://www.bishopsgatedental.co.uk/about-us/',
 'https://www.bishopsgatedental.co.uk/about-us/meet-our-team/',
 'https://www.bishopsgatedental.co.uk/about-us/award-winning-london-dentist/',
 'https://www.bishopsgatedental.co.uk/about-us/testimonials/',
 'https://www.bishopsgatedental.co.uk/blog/',
 'https://www.bishopsgatedental.co.uk/invisalign/',
 'https://www.bishopsgatedental.co.uk/invisalign/',
 'https://www.bishopsgatedental.co.uk/invisalign/before-after/',
 'https://www.bishopsgatedental.co.uk/invisalign/consultations/',
 'https://www.bishopsgatedental.co.uk/invisalign/treatment-cost/',
 'https://www.bishopsgatedental.co.uk/invisalign/invisalign-for-teens/',
 'https://www.bishopsgatedental.co.uk/invisalign/treatment-process/',
 'https://www.bishopsgatedental.co.u

In [72]:
link_system_prompt = """
You are an advisor for an enterprise that sells AI booking agents.

You are provided with a list of links to a company's website.

You are able to decide which of the links would be most relevant to include in an analysis and profiling about the company
assessing whether it is a good sales lead for your enterprise.

Select links that are likely to contain useful business information, such as:

- services
- treatments
- pricing or fees
- emergency services
- booking or appointment pages
- reviews or testimonials
- FAQs

You should respond in JSON as in this example:
{
    "links": [
        {"type": "prices", "url": "https://full.url/goes/here/pricing"},
        {"type": "offering", "url": "https://full.url/goes/here/offering"},
        {"type":"service", "url": "https://full.url/goes/here/service"}
    ]
"}

Ignore links that are not useful, such as:
- privacy policy
- cookie policy
- terms and conditions
- accessibility pages
- login pages
- admin pages
- basket, checkout, account, or cart pages
- blog posts, unless they clearly describe a core service
- image files, PDFs, scripts, feeds, tags, categories, or assets
- duplicate or near-duplicate links
- social media links"""

In [73]:
def get_link_user_prompt(url):
    link_user_prompt = f"""
    Here are the links found on the webpage {url}:

    Please decide which of these links are the most relevant to support an overall analysis and profiling of a company 
    assessing whether it is a good sales lead.
    """
    links = fetch_website_links(url)
    link_user_prompt += "\n".join(links)
    return link_user_prompt

In [76]:
print(get_link_user_prompt('https://www.bishopsgatedental.co.uk/'))


    Here are the links found on the webpage https://www.bishopsgatedental.co.uk/:

    Please decide which of these links are the most relevant to support an overall analysis and profiling of a company 
    assessing whether it is a good sales lead.
    #content
tel:02073776762
/contact-us/
tel:02073776762
https://uk.dentalhub.online/soe/new/Bishopsgate%20Dental%20Care?pid=UKMAD03
/contact-us/
https://www.bishopsgatedental.co.uk/about-us/
https://www.bishopsgatedental.co.uk/about-us/
https://www.bishopsgatedental.co.uk/about-us/meet-our-team/
https://www.bishopsgatedental.co.uk/about-us/award-winning-london-dentist/
https://www.bishopsgatedental.co.uk/about-us/testimonials/
https://www.bishopsgatedental.co.uk/blog/
https://www.bishopsgatedental.co.uk/invisalign/
https://www.bishopsgatedental.co.uk/invisalign/
https://www.bishopsgatedental.co.uk/invisalign/before-after/
https://www.bishopsgatedental.co.uk/invisalign/consultations/
https://www.bishopsgatedental.co.uk/invisalign/treatmen

In [78]:
openai = OpenAI()

def get_relevant_links(url):
    response = openai.chat.completions.create(
        model='gpt-4o-mini',
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_link_user_prompt(url)}
        ],
        response_format={"type":"json_object"}
    )
    results = response.choices[0].message.content
    links = json.loads(results)
    return links

get_relevant_links('https://www.bishopsgatedental.co.uk/')

{'links': [{'type': 'prices',
   'url': 'https://www.bishopsgatedental.co.uk/pricing/'},
  {'type': 'treatments',
   'url': 'https://www.bishopsgatedental.co.uk/invisalign/treatment-cost/'},
  {'type': 'services',
   'url': 'https://www.bishopsgatedental.co.uk/invisalign/'},
  {'type': 'services',
   'url': 'https://www.bishopsgatedental.co.uk/cosmetic-dentistry/'},
  {'type': 'services',
   'url': 'https://www.bishopsgatedental.co.uk/general-dentistry/'},
  {'type': 'emergency',
   'url': 'https://www.bishopsgatedental.co.uk/contact-us/'},
  {'type': 'reviews',
   'url': 'https://www.bishopsgatedental.co.uk/about-us/testimonials/'},
  {'type': 'faq',
   'url': 'https://www.bishopsgatedental.co.uk/invisalign/consultations/'}]}

In [85]:
def fetch_page_and_all_relevant_links(url):
    print(f'Fetching page and all relevant links from {url}')
    contents = fetch_website_contents(url)
    relevant_links = get_relevant_links(url)
    result = f'## Landing Page: \n\n {contents} \n\n ## Relevant Links: \n'
    for link in relevant_links['links']:
        result += f'## Link: {link['type']}\n'
        result += fetch_website_contents(link['url'])
    return result

In [87]:
print(fetch_page_and_all_relevant_links('https://www.harleystreetdentalstudio.com/'))

Fetching page and all relevant links from https://www.harleystreetdentalstudio.com/


## Landing Page: 

 Cosmetic Dentistry & Dentist London W1 - Harley Street Dental Studio

About
Meet The Team
Reviews
Our Practice
Awards
Smile Stories
Blog
Treatments
Cosmetic Dentistry
Cosmetic Dentistry
Veneers
Teeth Whitening
White Spot Treatment
Invisible Fillings
Composite Bonding
Cosmetic Dentures
Crowns & Bridges
Smile Makeovers
Gummy Smiles
Gum Grafting
Gum Disease
Tooth Sculpting
Digital Smile Design
Digital Dentistry
Tooth Wear Rehab
General Dentistry
General Dentistry
TMJ Treatment
Children's Dentistry
Oral Cancer Scanner
Hygiene
Headache Treatments
Dental Emergencies
Nervous Patient Care
Nervous Patient Care
No Drill Dentistry
Sedation
More
Sports Performance
Anti Snoring
Facial Cosmetic Treatments
Specialist Dentistry
Soprano Laser Hair Removal
Fat Freezing With Cooltech
Cellulite Removal With Velashape
Braces
Aligners
Invisalign
Invisalign Teen
Invisalign Lite
Simpli5
Inman Aligner
Braces
Damon Braces
6 Month Smiles
Incognito Lingual Braces
Stb Social 6 Lingual Braces
Cf

In [88]:
lead_qualifier_system_prompt = """
You are an advisor for an enterprise that sells AI booking agents in the dental industry - assessing whether a company is a good sales lead.

You analyze the contents of several relevant pages from a company website and creates a short profile of the company as a prospective customer.

The profile should include:
- Company Name
- Company Description
- Company Industry
- Potential Opportunities for the AI booking agent at company
- Potential Company Pain Points

Respond in markdown without code blocks.
"""

In [89]:
def get_lead_qualifier_user_prompt(company_name, url):
    user_prompt = f"""
    You are looking at a company called: {company_name}
    Here are the contents of its landing page and other relevant pages;
    use this information to build a short profile of the company in markdown without code blocks.
    
    """
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5000]
    return user_prompt

In [91]:
print(get_lead_qualifier_user_prompt('Harley Street Dental Studio', 'https://www.harleystreetdentalstudio.com/'))

Fetching page and all relevant links from https://www.harleystreetdentalstudio.com/

    You are looking at a company called: Harley Street Dental Studio
    Here are the contents of its landing page and other relevant pages;
    use this information to build a short profile of the company in markdown without code blocks.

    ## Landing Page: 

 Cosmetic Dentistry & Dentist London W1 - Harley Street Dental Studio

About
Meet The Team
Reviews
Our Practice
Awards
Smile Stories
Blog
Treatments
Cosmetic Dentistry
Cosmetic Dentistry
Veneers
Teeth Whitening
White Spot Treatment
Invisible Fillings
Composite Bonding
Cosmetic Dentures
Crowns & Bridges
Smile Makeovers
Gummy Smiles
Gum Grafting
Gum Disease
Tooth Sculpting
Digital Smile Design
Digital Dentistry
Tooth Wear Rehab
General Dentistry
General Dentistry
TMJ Treatment
Children's Dentistry
Oral Cancer Scanner
Hygiene
Headache Treatments
Dental Emergencies
Nervous Patient Care
Nervous Patient Care
No Drill Dentistry
Sedation
More
Sports Pe

In [93]:
def create_lead_qualifier(company_name, url):
    stream = openai.chat.completions.create(
        model = 'gpt-4.1-mini',
        messages = [
            {"role":"system", "content": lead_qualifier_system_prompt},
            {"role":"user", "content": get_lead_qualifier_user_prompt(company_name, url)}
        ],
        stream = True
    )
    response = ""
    display_handle = display(Markdown(response), display_id = True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id = display_handle.display_id)

In [94]:
create_lead_qualifier('Harley Street Dental Studio', 'https://www.harleystreetdentalstudio.com/')

Fetching page and all relevant links from https://www.harleystreetdentalstudio.com/


# Company Profile: Harley Street Dental Studio

**Company Name:**  
Harley Street Dental Studio

**Company Description:**  
Harley Street Dental Studio is a prestigious dental practice located in central London, renowned for delivering world-class, pain-free cosmetic and general dental care since 2004. The clinic boasts a carefully handpicked team of highly skilled dentists and specialists overseen by founding dentists, with a strong emphasis on excellence, attention to detail, and patient comfort. It offers a wide range of services including cosmetic dentistry (veneers, teeth whitening, smile makeovers), general dentistry, orthodontics (braces and aligners), dental implants, and specialist treatments such as sedation and nervous patient care. The practice also incorporates advanced digital dentistry and facial cosmetic treatments, positioning itself at the forefront of modern dental care.

**Company Industry:**  
Dental Services / Cosmetic and General Dentistry

**Potential Opportunities for the AI Booking Agent at Harley Street Dental Studio:**  
- Streamlining appointment scheduling for a broad range of dental and cosmetic treatments, reducing administrative burden.  
- Managing online bookings efficiently for high-demand treatments such as Invisalign, teeth whitening, and dental implants.  
- Handling patient inquiries, including treatment options and financing, 24/7 to improve patient engagement and conversion.  
- Supporting patient management for nervous or sedation patients who may require special scheduling consideration.  
- Integrating with their online booking system to optimize calendar management across multiple specialists.  
- Assisting in referral management and follow-up appointments, enhancing patient retention.

**Potential Company Pain Points:**  
- High patient volume and diverse treatment offerings could strain manual booking processes leading to scheduling conflicts or delays.  
- Managing appointment complexities for specialist treatments requiring coordination among expert clinicians.  
- Providing quick responses to patient queries outside office hours to maintain high standards of patient care and satisfaction.  
- Handling the needs of nervous patients who may require additional reassurance or custom scheduling solutions.  
- Ensuring seamless integration of booking solutions with their existing online and phone-based scheduling platforms.